# Optional · Resolution: convolution of a Breit–Wigner with a Gaussian
Lab 1 (Exercise B1) used `RooVoigtian`. Here we build the convolution **explicitly** with `RooFFTConvPdf`
and see what it does. Data: the $B^0_s$ mass histogram of Lab 1.

In [ ]:
import ROOT
import numpy as np
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

In [ ]:
f = ROOT.TFile.Open("../data/b0_invariant_mass.root")
h = f.Get("massaB0"); h.SetDirectory(0); f.Close()
mass = ROOT.RooRealVar("mass", "mass [GeV]", 5.0, 6.0)
dh = ROOT.RooDataHist("dh", "dh", ROOT.RooArgList(mass), h)

mean  = ROOT.RooRealVar("mean", "mean", 5.36, 5.2, 5.5)
width = ROOT.RooRealVar("width", "natural width", 0.05, 0.001, 0.5)
bw    = ROOT.RooBreitWigner("bw", "bw", mass, mean, width)

sigma_res = ROOT.RooRealVar("sigma_res", "resolution", 0.03)      # fixed, from a resolution study
zero = ROOT.RooRealVar("zero", "zero", 0.0)
res  = ROOT.RooGaussian("res", "res", mass, zero, sigma_res)

mass.setBins(10000, "cache")                  # fine binning for the FFT
conv = ROOT.RooFFTConvPdf("conv", "BW (x) Gauss", mass, bw, res)
fit = conv.fitTo(dh, ROOT.RooFit.Save(), ROOT.RooFit.PrintLevel(-1))
print("mass = %.4f, width = %.4f" % (mean.getVal(), width.getVal()))

In [ ]:
frame = mass.frame(ROOT.RooFit.Title("BW (x) Gaussian"))
dh.plotOn(frame); conv.plotOn(frame)
c = ROOT.TCanvas("c", "c", 700, 500); frame.Draw(); c.Draw()
check("fit converged", fit.status() == 0)
check("mass within 30 MeV of 5.367 GeV", abs(mean.getVal() - 5.367) < 0.03)

## Questions
1. Compare with `RooVoigtian`: same result? Why is the FFT convolution more general?
2. Float `sigma_res` together with `width`. What happens to the errors and the correlation? What do real analyses do?
3. Convolve with a **double Gaussian** or a Crystal Ball instead. Which one models the tails better?